# Qwen 4B в Google Colab
Самостоятельный запуск весов **Qwen/Qwen3-4B-Instruct-2507** на GPU Colab, без API-ключей. Сначала выбери **Среда выполнения → Сменить среду выполнения → T4 GPU**. Выполняй ячейки сверху вниз.

Ответы после каждого вопроса сохраняются в `Мой диск/Lab3_4/answers_local.jsonl`. При обрыве подключения снова выполни ячейки: завершённые вопросы будут пропущены. Эталонные ответы из CSV модели не передаются.

Модель: https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507. Бесплатный GPU зависит от доступности: https://research.google.com/colaboratory/faq.html.

In [ ]:
%pip install -q "transformers==4.57.1" "accelerate>=1.0,<2" safetensors


## 1. Датасет и сохранение
Разреши подключение Google Диска и выбери **questions.csv** из `data/` из папки проекта.

In [ ]:
import csv
import json
import time
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')
FOLDER = Path('/content/drive/MyDrive/Lab3_4')
FOLDER.mkdir(exist_ok=True)
uploaded = files.upload()
(FOLDER / 'dataset.csv').write_bytes(next(iter(uploaded.values())))
with (FOLDER / 'dataset.csv').open(encoding='utf-8-sig', newline='') as file:
    questions = list(csv.DictReader(file))
LOG = FOLDER / 'answers_local.jsonl'
records = [json.loads(line) for line in LOG.read_text().splitlines()] if LOG.exists() else []
print('Вопросов:', len(questions))


## 2. Загрузка модели
Первый запуск скачает несколько гигабайт весов. FP16 подходит для T4; квантование не используется.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

if not torch.cuda.is_available():
    raise RuntimeError('Включи GPU: Среда выполнения → Сменить среду выполнения → T4 GPU.')

MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'
set_seed(42)
started = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map={'': 'cuda:0'},
    attn_implementation='sdpa', low_cpu_mem_usage=True,
).eval()
torch.cuda.synchronize()
ENV = {'gpu': torch.cuda.get_device_name(0), 'dtype': 'float16',
       'revision': model.config._commit_hash, 'seed': 42,
       'load_seconds': round(time.perf_counter() - started, 3),
       'libraries': {p: version(p) for p in ['torch', 'transformers', 'accelerate']}}
print(MODEL_ID, ENV['gpu'], f"{torch.cuda.memory_allocated() / 2**30:.2f} ГиБ занято весами")


## 3. Запрос к модели
Одинаковая системная инструкция и форматы ответов с API-тетрадью. `do_sample=False` включает жадную генерацию без случайного выбора токенов.


In [ ]:
SYSTEM = 'Решай задачи по машинному обучению на русском. Отвечай кратко, строго в указанном формате.'
FORMATS = {'A': 'Расчёт в 1–3 строках, затем ОТВЕТ: и все запрошенные значения. Без Markdown и LaTeX.',
 'B': 'Ответь «верно» или «неверно» и обоснуй одним коротким предложением.',
 'C': 'Назови метод и необходимые шаги; обоснуй выбор. Не более трёх коротких предложений.',
 'D': 'Только компактный исполняемый Python-код с импортами. Выполни все требования задачи. Без '
      'Markdown, комментариев и пояснений.',
 'E': 'Только метка класса, указанная в условии. Без пояснений.',
 'F': 'Короткий ответ с обоснованием, до трёх предложений. Если данных недостаточно, прямо укажи '
      'это.'}
MAX_TOKENS = {'A': 512, 'B': 256, 'C': 512, 'D': 1536, 'E': 32, 'F': 512}

def ask(prompt, question_id='', question_type=''):
    messages = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt}]
    limit = MAX_TOKENS.get(question_type, 512)
    record = {'date': datetime.now(timezone.utc).isoformat(), 'id': question_id,
              'тип': question_type, 'model': 'local', 'model_id': MODEL_ID,
              'request': {'model': MODEL_ID, 'messages': messages,
                          'do_sample': False, 'max_new_tokens': limit}, **ENV}
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    try:
        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=limit, do_sample=False,
                                    pad_token_id=tokenizer.pad_token_id)
        torch.cuda.synchronize()
        new_tokens = output[0, inputs['input_ids'].shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        eos = model.generation_config.eos_token_id
        eos = [eos] if isinstance(eos, int) else eos
        complete = len(new_tokens) > 0 and new_tokens[-1].item() in eos
        record.update(answer=answer, prompt_tokens=inputs['input_ids'].shape[1],
                      completion_tokens=len(new_tokens), tokens=output.shape[1],
                      finish_reason='stop' if complete else 'length')
        if not complete or not answer:
            raise ValueError('Нет полного ответа. Увеличь MAX_TOKENS для этого типа и повтори прогон.')
        return answer
    except (Exception, KeyboardInterrupt) as error:
        record['error'] = str(error) or 'Выполнение прервано'
        raise
    finally:
        record.update(seconds=round(time.perf_counter() - started, 3),
                      peak_gpu_gib=round(torch.cuda.max_memory_allocated() / 2**30, 3))
        if question_id:
            with LOG.open('a', encoding='utf-8') as file:
                file.write(json.dumps(record, ensure_ascii=False) + '\n')
            records.append(record)


In [ ]:
# Необязательный запрос: раскомментируй для проверки.
# print(ask('Чем precision отличается от recall?'))


## 4. Весь датасет
Сохраняются текст ответа, параметры, токены, время и пиковая память GPU. Повторный запуск пропускает полные ответы для той же формулировки вопроса.

In [ ]:
def save_csv():
    by_id = {q['id']: q for q in questions}
    latest = {r['id']: r for r in records if r['id'] in by_id
              and r['request']['messages'][-1]['content'] == FORMATS[by_id[r['id']]['тип']] + '\n\n' + by_id[r['id']]['текст_ru']}
    with (FOLDER / 'answers_local.csv').open('w', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=['id', 'тип', 'model', 'вопрос', 'ответ', 'ошибка'],
                                lineterminator='\n')
        writer.writeheader()
        for qid, r in latest.items():
            row = {'id': qid, 'тип': by_id[qid]['тип'], 'model': 'local',
                   'вопрос': by_id[qid]['текст_ru'], 'ответ': r.get('answer', ''), 'ошибка': r.get('error', '')}
            writer.writerow({k: v.replace('\r\n', '\n').replace('\r', '\n').replace('\n', r'\n') for k, v in row.items()})

for question in questions:
    prompt = FORMATS[question['тип']] + '\n\n' + question['текст_ru']
    if any(r['id'] == question['id'] and r.get('answer') and not r.get('error')
           and r.get('finish_reason') != 'length' and r['request']['messages'][-1]['content'] == prompt
           for r in records):
        print(question['id'], 'уже готово')
        continue
    try:
        ask(prompt, question['id'], question['тип'])
        print(question['id'], 'готово')
    except Exception as error:
        print(question['id'], error)
    finally:
        save_csv()

save_csv()
print('Результаты:', FOLDER)


## 5. Перенос в VS Code
Скачай `local_responses.jsonl` и положи в папку `logs/` проекта. Выполни `collect_api_answers.ipynb`, затем `analyze_answers.ipynb`: готовые API-ответы пропускаются, таблицы обновляются без повторных запросов.

In [ ]:
export = Path('/content/local_responses.jsonl')
export.write_bytes(LOG.read_bytes())
files.download(str(export))
